In [4]:
%env CUDA_VISIBLE_DEVICES=1

env: CUDA_VISIBLE_DEVICES=1


In [5]:
from xvla_wlr.agent import XVLAAgent, XVLAAction, XVLAObservation, XVLA_DOMAIN_IDS


In [6]:
# import accelerate

# accelerator = accelerate.Accelerator(
#     mixed_precision="bf16",
#     dynamo_plugin=accelerate.utils.TorchDynamoPlugin(
#         backend="inductor",
#         mode="reduce-overhead",
#         fullgraph=True,
#         # dynamic=True,
#     )
# )

In [7]:
import torch

torch.set_float32_matmul_precision("high")

agent = XVLAAgent(
    "/home/ace/X-VLA/workspaces/experiment_sample/checkpoints/current/checkpoint.json",
    accelerator=True,
    dtype=torch.bfloat16,
)

Florence2ForConditionalGeneration has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


In [8]:

torch.get_float32_matmul_precision()

'high'

In [ ]:
# %%timeit -n 10
import torch

# compute_actions = torch.compile(agent.compute_actions, mode="max-autotune", disable=False)
compute_actions = agent.compute_actions
%timeit -n 10 
compute_actions(XVLAObservation.sample())

In [15]:
import torch

compute_actions = torch.compile(agent.compute_actions, mode="max-autotune")
with torch.profiler.profile() as prof:
    compute_actions(XVLAObservation.sample())
prof.export_chrome_trace("./trace-v7.json")

In [ ]:
# accelerator.autocast?

Signature: accelerator.autocast(autocast_handler: 'AutocastKwargs' = None)
Docstring:
Will apply automatic mixed-precision inside the block inside this context manager, if it is enabled. Nothing
different will happen otherwise.

A different `autocast_handler` can be passed in to override the one set in the `Accelerator` object. This is
useful in blocks under `autocast` where you want to revert to fp32.

Example:

```python
>>> from accelerate import Accelerator

>>> accelerator = Accelerator(mixed_precision="fp16")
>>> with accelerator.autocast():
...     train()
```
File:      ~/X-VLA/.conda/lib/python3.11/site-packages/accelerate/accelerator.py
Type:      method

In [ ]:
accelerator

In [9]:
%%timeit -n 1

import torch

with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
    agent.compute_actions(XVLAObservation.sample())


105 ms ± 5.96 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [10]:
%%timeit -n 1
agent.compute_actions(XVLAObservation.sample())

137 ms ± 1.83 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
